<a href="https://colab.research.google.com/github/mebre7/house-price-predictor-mlops/blob/colab_branch/notebooks/01_eda_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploratory Data Analysis (EDA)
Exploratory Data Analysis (EDA) is a crucial step in the data analysis process that involves examining and visualizing data to uncover patterns, relationships, and insights. It helps in understanding the underlying structure of the data, identifying anomalies, and informing subsequent modeling decisions.

## 1. Load Data from PostgreSQL

In [50]:
from sqlalchemy import create_engine, text
import pandas as pd
import os
from dotenv import load_dotenv
from urllib.parse import quote_plus

In [51]:
load_dotenv()

DB_USER = os.getenv("DB_USER", "")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD", ""))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_NAME = os.getenv("DB_NAME", "")

# 2. Build SQLAlchemy connection string
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(DATABASE_URL)
with engine.connect() as conn:
    df = pd.read_sql(
        text("SELECT * FROM housing_data"),
        conn
    )

df.head()

,Order,PID,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


### Import packages

In [52]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

**Change column names to snake_case for consistency and ease of use.**

In [53]:
df = df.rename(columns=lambda x: x.strip().replace(" ", "_").replace("-", "_"))

In [54]:
df

,Order,PID,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,2926,923275080,80,RL,37.0,7937,Pave,NaN,IR1,Lvl,...,0,NaN,GdPrv,NaN,0,3,2006,WD,Normal,142500
2926,2927,923276100,20,RL,NaN,8885,Pave,NaN,IR1,Low,...,0,NaN,MnPrv,NaN,0,6,2006,WD,Normal,131000
2927,2928,923400125,85,RL,62.0,10441,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,Shed,700,7,2006,WD,Normal,132000
2928,2929,924100070,20,RL,77.0,10010,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2006,WD,Normal,170000


## 2. Basic Shape & Types Audit

In [55]:
df.shape

(2930, 82)

-> $2930$ rows and $82$ features/columns

In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS_SubClass      2930 non-null   int64  
 3   MS_Zoning        2930 non-null   str    
 4   Lot_Frontage     2440 non-null   float64
 5   Lot_Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot_Shape        2930 non-null   str    
 9   Land_Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot_Config       2930 non-null   str    
 12  Land_Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition_1      2930 non-null   str    
 15  Condition_2      2930 non-null   str    
 16  Bldg_Type        2930 non-null   str    
 17  House_Style      2930 non

Of 82 columns:
- Float: 11
- Integer: 28
- Object (string): 43

In [57]:
df.dtypes

Order               int64
PID                 int64
MS_SubClass         int64
MS_Zoning             str
Lot_Frontage      float64
                   ...   
Mo_Sold             int64
Yr_Sold             int64
Sale_Type             str
Sale_Condition        str
SalePrice           int64
Length: 82, dtype: object

In [58]:
df.describe()

,Order,PID,MS_SubClass,Lot_Frontage,Lot_Area,Overall_Qual,Overall_Cond,Year_Built,Year_Remod/Add,Mas_Vnr_Area,...,Wood_Deck_SF,Open_Porch_SF,Enclosed_Porch,3Ssn_Porch,Screen_Porch,Pool_Area,Misc_Val,Mo_Sold,Yr_Sold,SalePrice
count,2930.00000,2.930000e+03,2930.000000,2440.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2907.000000,...,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000,2930.000000
mean,1465.50000,7.144645e+08,57.387372,69.224590,10147.921843,6.094881,5.563140,1971.356314,1984.266553,101.896801,...,93.751877,47.533447,23.011604,2.592491,16.002048,2.243345,50.635154,6.216041,2007.790444,180796.060068
std,845.96247,1.887308e+08,42.638025,23.365335,7880.017759,1.411026,1.111537,30.245361,20.860286,179.112611,...,126.361562,67.483400,64.139059,25.141331,56.087370,35.597181,566.344288,2.714492,1.316613,79886.692357
min,1.00000,5.263011e+08,20.000000,21.000000,1300.000000,1.000000,1.000000,1872.000000,1950.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,2006.000000,12789.000000
25%,733.25000,5.284770e+08,20.000000,58.000000,7440.250000,5.000000,5.000000,1954.000000,1965.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,4.000000,2007.000000,129500.000000
50%,1465.50000,5.354536e+08,50.000000,68.000000,9436.500000,6.000000,5.000000,1973.000000,1993.000000,0.000000,...,0.000000,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.000000,2008.000000,160000.000000
75%,2197.75000,9.071811e+08,70.000000,80.000000,11555.250000,7.000000,6.000000,2001.000000,2004.000000,164.000000,...,168.000000,70.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,2009.000000,213500.000000
max,2930.00000,1.007100e+09,190.000000,313.000000,215245.000000,10.000000,9.000000,2010.000000,2010.000000,1600.000000,...,1424.000000,742.000000,1012.000000,508.000000,576.000000,800.000000,17000.000000,12.000000,2010.000000,755000.000000


Drop the following columns:

- `Order` as it is just a row number and does not provide any useful information for analysis.
- `PID` as it is a unique identifier for each property and does not contribute to the analysis.

In [59]:
PID = df['PID']
df = df.drop(['PID', 'Order'], axis=1)

In [61]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


### Numerical and Categorical Columns

In [178]:
# define numerical & categorical columns
numerical_features = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = df.select_dtypes(include=[object]).columns.tolist()
all_features = list(numerical_features) + list(categorical_features)

In [179]:
# print columns
print(f'{len(numerical_features)} numerical features: {numerical_features}')
print(f'\n{len(categorical_features)} categorical features: {categorical_features}')

37 numerical features: ['MS_SubClass', 'Lot_Frontage', 'Lot_Area', 'Overall_Qual', 'Overall_Cond', 'Year_Built', 'Year_Remod/Add', 'Mas_Vnr_Area', 'BsmtFin_SF_1', 'BsmtFin_SF_2', 'Bsmt_Unf_SF', 'Total_Bsmt_SF', '1st_Flr_SF', '2nd_Flr_SF', 'Low_Qual_Fin_SF', 'Gr_Liv_Area', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Yr_Blt', 'Garage_Cars', 'Garage_Area', 'Wood_Deck_SF', 'Open_Porch_SF', 'Enclosed_Porch', '3Ssn_Porch', 'Screen_Porch', 'Pool_Area', 'Misc_Val', 'Mo_Sold', 'Yr_Sold', 'SalePrice']

43 categorical features: ['MS_Zoning', 'Street', 'Alley', 'Lot_Shape', 'Land_Contour', 'Utilities', 'Lot_Config', 'Land_Slope', 'Neighborhood', 'Condition_1', 'Condition_2', 'Bldg_Type', 'House_Style', 'Roof_Style', 'Roof_Matl', 'Exterior_1st', 'Exterior_2nd', 'Mas_Vnr_Type', 'Exter_Qual', 'Exter_Cond', 'Foundation', 'Bsmt_Qual', 'Bsmt_Cond', 'Bsmt_Exposure', 'BsmtFin_Type_1', 'BsmtFin_Type_2', 'Heating', 'H

There are:
- **37** numeric features
- **43** categorical features

#### Frequency of each column
- Proportion of individual values in each column

In [116]:
for col in categorical_features:
  print(df[col].value_counts(normalize=True) * 100)
  print(f"\n----------------------------------")

MS_Zoning
RL         77.576792
RM         15.767918
FV          4.744027
RH          0.921502
C (all)     0.853242
I (all)     0.068259
A (agr)     0.068259
Name: proportion, dtype: float64

----------------------------------
Street
Pave    99.590444
Grvl     0.409556
Name: proportion, dtype: float64

----------------------------------
Alley
Grvl    60.606061
Pave    39.393939
Name: proportion, dtype: float64

----------------------------------
Lot_Shape
Reg    63.447099
IR1    33.412969
IR2     2.593857
IR3     0.546075
Name: proportion, dtype: float64

----------------------------------
Land_Contour
Lvl    89.863481
HLS     4.095563
Bnk     3.993174
Low     2.047782
Name: proportion, dtype: float64

----------------------------------
Utilities
AllPub    99.897611
NoSewr     0.068259
NoSeWa     0.034130
Name: proportion, dtype: float64

----------------------------------
Lot_Config
Inside     73.037543
Corner     17.440273
CulDSac     6.143345
FR2         2.901024
FR3         0.477816

## 3. Missing Value Analysis

In [117]:
df.isnull().sum().sort_values(ascending=False)

Pool_QC           2917
Misc_Feature      2824
Alley             2732
Fence             2358
Mas_Vnr_Type      1775
                  ... 
Mo_Sold              0
Yr_Sold              0
Sale_Type            0
Sale_Condition       0
SalePrice            0
Length: 80, dtype: int64

In [118]:
# count the number of rows in the dataframe in each column that have null values
df.isnull().count()

MS_SubClass       2930
MS_Zoning         2930
Lot_Frontage      2930
Lot_Area          2930
Street            2930
                  ... 
Mo_Sold           2930
Yr_Sold           2930
Sale_Type         2930
Sale_Condition    2930
SalePrice         2930
Length: 80, dtype: int64

**Count the number of missing values in each column and percentage of missing values in each column. This helps to identify which columns have a significant amount of missing data and may require imputation or removal.**

In [119]:
count = df.isnull().sum().sort_values(ascending=False)
percentage = ((df.isnull().sum() / df.isnull().count()) * 100).sort_values(ascending=False)
null_values = pd.concat([count, percentage], axis=1, keys=['Count', 'Percentage'])

In [120]:
null_values.head(n=28)

,Count,Percentage
Pool_QC,2917,99.556314
Misc_Feature,2824,96.382253
Alley,2732,93.242321
Fence,2358,80.477816
Mas_Vnr_Type,1775,60.580205
Fireplace_Qu,1422,48.532423
Lot_Frontage,490,16.723549
Garage_Yr_Blt,159,5.426621
Garage_Qual,159,5.426621
Garage_Cond,159,5.426621


**Insights**:
- The `PoolQC` column has the highest percentage of missing values (99.55%), followed by `MiscFeature` (96.38%) and `Alley` (93.24%). These columns may require special attention for imputation or removal.

**How many columns have missing values?**

In [121]:
# how many columns have missing values?
columns_with_missing_values = null_values[null_values['Count'] > 0].shape[0]
columns_with_missing_values

27

> There are $27$ columns with missing values. Remaining $53$ columns have no missing values.

**Any duplicated rows?**

In [122]:
df.duplicated().sum()

np.int64(0)

> No duplicated rows were found in the dataset, indicating that each row represents a unique property listing.

## 4. Feature Classification

1. **Categorical Features**: Columns that represent distinct categories or groups.
2. **Numerical Features**: Columns that represent numbers. These are further divided into continuous and discrete:
   - **Discrete Features**: Numerical values that are countable and finite.
   - **Continuous Features**: Numerical values that can take any value.
3. **Temporal Features**: Features that represent time.

In [123]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


**1. Numerical features**

In [124]:
# Numerical features
numerical_features = numerical_features
# Categorical features
categorical_features = categorical_features
# All features
all_features = numerical_features + categorical_features

In [125]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [126]:
# Discrete features
discrete_num_features = ['MS_SubClass', 'Overall_Qual', 'Overall_Cond', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Cars']
discrete_num_features

['MS_SubClass',
 'Overall_Qual',
 'Overall_Cond',
 'Bsmt_Full_Bath',
 'Bsmt_Half_Bath',
 'Full_Bath',
 'Half_Bath',
 'Bedroom_AbvGr',
 'Kitchen_AbvGr',
 'TotRms_AbvGrd',
 'Fireplaces',
 'Garage_Cars']

In [149]:
# Add discrete features to categorical features list
categorical_features_all = list(set(categorical_features + discrete_num_features))

> Add discrete features to categorical features list. This is done because discrete features, while numerical, often represent categories (e.g., number of bedrooms) and can be treated as categorical for analysis purposes.

**2. Categorical features**

In [151]:
categorical_features_all

['Condition_1',
 'MS_SubClass',
 'Bsmt_Exposure',
 'Paved_Drive',
 'Electrical',
 'Exter_Qual',
 'Sale_Condition',
 'Mas_Vnr_Type',
 'Garage_Finish',
 'Bsmt_Cond',
 'Street',
 'Bsmt_Full_Bath',
 'Neighborhood',
 'Foundation',
 'Heating',
 'Functional',
 'Bedroom_AbvGr',
 'Roof_Style',
 'Roof_Matl',
 'House_Style',
 'Pool_QC',
 'Sale_Type',
 'Kitchen_Qual',
 'Lot_Config',
 'Garage_Cars',
 'Half_Bath',
 'Overall_Qual',
 'Bsmt_Qual',
 'Land_Slope',
 'Fence',
 'Overall_Cond',
 'Bldg_Type',
 'MS_Zoning',
 'Condition_2',
 'Fireplaces',
 'BsmtFin_Type_1',
 'Garage_Cond',
 'Bsmt_Half_Bath',
 'BsmtFin_Type_2',
 'Garage_Qual',
 'TotRms_AbvGrd',
 'Central_Air',
 'Exterior_2nd',
 'Garage_Type',
 'Fireplace_Qu',
 'Lot_Shape',
 'Exterior_1st',
 'Kitchen_AbvGr',
 'Heating_QC',
 'Alley',
 'Misc_Feature',
 'Exter_Cond',
 'Full_Bath',
 'Utilities',
 'Land_Contour']

In [152]:
# Continuous features
continuous_num_features = list(set(numerical_features) - set(discrete_num_features))
continuous_num_features

['Year_Remod/Add',
 '1st_Flr_SF',
 'Bsmt_Unf_SF',
 'Wood_Deck_SF',
 'Year_Built',
 '3Ssn_Porch',
 '2nd_Flr_SF',
 'Low_Qual_Fin_SF',
 'Gr_Liv_Area',
 'Open_Porch_SF',
 'Yr_Sold',
 'BsmtFin_SF_1',
 'Mo_Sold',
 'Total_Bsmt_SF',
 'BsmtFin_SF_2',
 'Pool_Area',
 'Lot_Frontage',
 'Garage_Yr_Blt',
 'Screen_Porch',
 'Enclosed_Porch',
 'Garage_Area',
 'Lot_Area',
 'Mas_Vnr_Area',
 'SalePrice',
 'Misc_Val']

In [153]:
df.head()

,MS_SubClass,MS_Zoning,Lot_Frontage,Lot_Area,Street,Alley,Lot_Shape,Land_Contour,Utilities,Lot_Config,...,Pool_Area,Pool_QC,Fence,Misc_Feature,Misc_Val,Mo_Sold,Yr_Sold,Sale_Type,Sale_Condition,SalePrice
0,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,Corner,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,AllPub,Corner,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,Inside,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [154]:
date_year = ['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', ]
date_month = ['Mo_Sold']

In [155]:
for col in date_year:
    df[col] = pd.to_datetime(df[col], format='%Y').dt.year

for col in date_month:
    df[col] = pd.to_datetime(df[col], format='%m').dt.month

In [156]:
# Temporal features
temporal_features = date_year + date_month
temporal_features

['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', 'Mo_Sold']

1. Numerical features

In [157]:
discrete_num_features = discrete_num_features
continuous_num_features = continuous_num_features

In [158]:
print(f"{len(discrete_num_features)} discrete numerical features: {discrete_num_features}\n")
print(f"{len(continuous_num_features)} continuous numerical features: {continuous_num_features}")

12 discrete numerical features: ['MS_SubClass', 'Overall_Qual', 'Overall_Cond', 'Bsmt_Full_Bath', 'Bsmt_Half_Bath', 'Full_Bath', 'Half_Bath', 'Bedroom_AbvGr', 'Kitchen_AbvGr', 'TotRms_AbvGrd', 'Fireplaces', 'Garage_Cars']

25 continuous numerical features: ['Year_Remod/Add', '1st_Flr_SF', 'Bsmt_Unf_SF', 'Wood_Deck_SF', 'Year_Built', '3Ssn_Porch', '2nd_Flr_SF', 'Low_Qual_Fin_SF', 'Gr_Liv_Area', 'Open_Porch_SF', 'Yr_Sold', 'BsmtFin_SF_1', 'Mo_Sold', 'Total_Bsmt_SF', 'BsmtFin_SF_2', 'Pool_Area', 'Lot_Frontage', 'Garage_Yr_Blt', 'Screen_Porch', 'Enclosed_Porch', 'Garage_Area', 'Lot_Area', 'Mas_Vnr_Area', 'SalePrice', 'Misc_Val']


2. Categorical features

In [159]:
categorical_features_all = categorical_features_all

In [160]:
print(f"{len(categorical_features_all)} categorical features: {categorical_features_all}")

55 categorical features: ['Condition_1', 'MS_SubClass', 'Bsmt_Exposure', 'Paved_Drive', 'Electrical', 'Exter_Qual', 'Sale_Condition', 'Mas_Vnr_Type', 'Garage_Finish', 'Bsmt_Cond', 'Street', 'Bsmt_Full_Bath', 'Neighborhood', 'Foundation', 'Heating', 'Functional', 'Bedroom_AbvGr', 'Roof_Style', 'Roof_Matl', 'House_Style', 'Pool_QC', 'Sale_Type', 'Kitchen_Qual', 'Lot_Config', 'Garage_Cars', 'Half_Bath', 'Overall_Qual', 'Bsmt_Qual', 'Land_Slope', 'Fence', 'Overall_Cond', 'Bldg_Type', 'MS_Zoning', 'Condition_2', 'Fireplaces', 'BsmtFin_Type_1', 'Garage_Cond', 'Bsmt_Half_Bath', 'BsmtFin_Type_2', 'Garage_Qual', 'TotRms_AbvGrd', 'Central_Air', 'Exterior_2nd', 'Garage_Type', 'Fireplace_Qu', 'Lot_Shape', 'Exterior_1st', 'Kitchen_AbvGr', 'Heating_QC', 'Alley', 'Misc_Feature', 'Exter_Cond', 'Full_Bath', 'Utilities', 'Land_Contour']


3. Temporal features

In [161]:
temporal_features = temporal_features

In [162]:
print(f"{len(temporal_features)} temporal features: {temporal_features}")

5 temporal features: ['Year_Built', 'Year_Remod/Add', 'Garage_Yr_Blt', 'Yr_Sold', 'Mo_Sold']


In [163]:
print(f"Number of numerical features: {len(numerical_features)}")
print(f"Number of categorical features: {len(categorical_features_all)}")
print(f"Number of continuous numerical features: {len(continuous_num_features)}")
print(f"Number of discrete numerical features: {len(discrete_num_features)}")
print(f"Number of all features: {len(all_features)}")

Number of numerical features: 37
Number of categorical features: 55
Number of continuous numerical features: 25
Number of discrete numerical features: 12
Number of all features: 80


In [166]:
df[numerical_features].info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 37 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   MS_SubClass      2930 non-null   int64  
 1   Lot_Frontage     2440 non-null   float64
 2   Lot_Area         2930 non-null   int64  
 3   Overall_Qual     2930 non-null   int64  
 4   Overall_Cond     2930 non-null   int64  
 5   Year_Built       2930 non-null   int32  
 6   Year_Remod/Add   2930 non-null   int32  
 7   Mas_Vnr_Area     2907 non-null   float64
 8   BsmtFin_SF_1     2929 non-null   float64
 9   BsmtFin_SF_2     2929 non-null   float64
 10  Bsmt_Unf_SF      2929 non-null   float64
 11  Total_Bsmt_SF    2929 non-null   float64
 12  1st_Flr_SF       2930 non-null   int64  
 13  2nd_Flr_SF       2930 non-null   int64  
 14  Low_Qual_Fin_SF  2930 non-null   int64  
 15  Gr_Liv_Area      2930 non-null   int64  
 16  Bsmt_Full_Bath   2928 non-null   float64
 17  Bsmt_Half_Bath   2928 non

In [181]:
df[categorical_features_all].info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 55 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   Condition_1     2930 non-null   category
 1   MS_SubClass     2930 non-null   category
 2   Bsmt_Exposure   2847 non-null   category
 3   Paved_Drive     2930 non-null   category
 4   Electrical      2929 non-null   category
 5   Exter_Qual      2930 non-null   category
 6   Sale_Condition  2930 non-null   category
 7   Mas_Vnr_Type    1155 non-null   category
 8   Garage_Finish   2771 non-null   category
 9   Bsmt_Cond       2850 non-null   category
 10  Street          2930 non-null   category
 11  Bsmt_Full_Bath  2928 non-null   category
 12  Neighborhood    2930 non-null   category
 13  Foundation      2930 non-null   category
 14  Heating         2930 non-null   category
 15  Functional      2930 non-null   category
 16  Bedroom_AbvGr   2930 non-null   category
 17  Roof_Style      2930 non-

**Change data types of `categorical_features` to `category` for memory efficiency and better performance in modeling.**

In [182]:
for col in categorical_features_all:
    df[col] = df[col].astype('category')

In [183]:
df[categorical_features_all].info()

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 55 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   Condition_1     2930 non-null   category
 1   MS_SubClass     2930 non-null   category
 2   Bsmt_Exposure   2847 non-null   category
 3   Paved_Drive     2930 non-null   category
 4   Electrical      2929 non-null   category
 5   Exter_Qual      2930 non-null   category
 6   Sale_Condition  2930 non-null   category
 7   Mas_Vnr_Type    1155 non-null   category
 8   Garage_Finish   2771 non-null   category
 9   Bsmt_Cond       2850 non-null   category
 10  Street          2930 non-null   category
 11  Bsmt_Full_Bath  2928 non-null   category
 12  Neighborhood    2930 non-null   category
 13  Foundation      2930 non-null   category
 14  Heating         2930 non-null   category
 15  Functional      2930 non-null   category
 16  Bedroom_AbvGr   2930 non-null   category
 17  Roof_Style      2930 non-

> Now data types of categorical features have been changed to `category`.

In [187]:
# Describe the categorical features
df[categorical_features_all].describe()

,Condition_1,MS_SubClass,Bsmt_Exposure,Paved_Drive,Electrical,Exter_Qual,Sale_Condition,Mas_Vnr_Type,Garage_Finish,Bsmt_Cond,...,Lot_Shape,Exterior_1st,Kitchen_AbvGr,Heating_QC,Alley,Misc_Feature,Exter_Cond,Full_Bath,Utilities,Land_Contour
count,2930,2930,2847,2930,2929,2930,2930,1155,2771,2850,...,2930,2930,2930,2930,198,106,2930,2930,2930,2930
unique,9,16,4,3,5,4,6,4,3,5,...,4,16,4,5,2,5,5,5,3,4
top,Norm,20,No,Y,SBrkr,TA,Normal,BrkFace,Unf,TA,...,Reg,VinylSd,1,Ex,Grvl,Shed,TA,2,AllPub,Lvl
freq,2522,1079,1906,2652,2682,1799,2413,880,1231,2616,...,1859,1026,2796,1495,120,95,2549,1532,2927,2633


**Exclude `SalePrice` from the list of numerical features.** 

In [189]:
numerical_features = [col for col in numerical_features if col != 'SalePrice']
numerical_features

['MS_SubClass',
 'Lot_Frontage',
 'Lot_Area',
 'Overall_Qual',
 'Overall_Cond',
 'Year_Built',
 'Year_Remod/Add',
 'Mas_Vnr_Area',
 'BsmtFin_SF_1',
 'BsmtFin_SF_2',
 'Bsmt_Unf_SF',
 'Total_Bsmt_SF',
 '1st_Flr_SF',
 '2nd_Flr_SF',
 'Low_Qual_Fin_SF',
 'Gr_Liv_Area',
 'Bsmt_Full_Bath',
 'Bsmt_Half_Bath',
 'Full_Bath',
 'Half_Bath',
 'Bedroom_AbvGr',
 'Kitchen_AbvGr',
 'TotRms_AbvGrd',
 'Fireplaces',
 'Garage_Yr_Blt',
 'Garage_Cars',
 'Garage_Area',
 'Wood_Deck_SF',
 'Open_Porch_SF',
 'Enclosed_Porch',
 '3Ssn_Porch',
 'Screen_Porch',
 'Pool_Area',
 'Misc_Val',
 'Mo_Sold',
 'Yr_Sold']